# IOAI — 2025 Contest Tricy Table Data (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, glob, zipfile
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train_tables.csv'):
    os.environ['KAGGLE_API_TOKEN'] = 'KGAT_5dd261fded8e0d7eb2c29d8d65dfabea'  # 내장 토큰(규칙 수락된 계정)
    os.system('pip -q install kaggle')
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    api.competition_download_files('neoai-2025-tricy-table-data', path='data', quiet=False)
    for zp in glob.glob('data/*.zip'):
        with zipfile.ZipFile(zp) as zf: zf.extractall('data')
        os.remove(zp)
print('데이터 준비:', 'data/train_tables.csv' if os.path.exists('data/train_tables.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# NeoAI 2025 — Tricy Table Data ("These tables are weird")

> **Northern Eurasia Olympiad in AI 2025 · Kaggle playground competition (회귀)**

표 데이터로 **`target`(연속값, ≈137–866)** 를 예측합니다. 특징은 `feat_0..feat_8`(수치) + `day,hour,minute`.
**"테이블이 이상하다"의 정체 = 모든 컬럼에 약 10% 결측값**이 흩뿌려져 있다는 점입니다(특히 `feat_1` 은
target 과 상관 0.996 으로 거의 결정적이지만 8.5% 가 비어 있음). 지표 = **RMSE**(낮을수록 좋음).

공식 Kaggle test 라벨은 비공개라 `train_tables.csv` 의 **결정적 held-out(행 index % 5 == 0)** 으로 채점합니다.
**제출** `submission.csv`(`id,target`) → 사이트 **Submissions** 탭에서 RMSE 채점.

⚠️ **아래 베이스라인은 결측을 그대로 둔 채 HistGradientBoosting(NaN 네이티브 처리)에 넣은 단순 모델(RMSE ≈ 6.2).**
결측을 *복원*(반복 대치)하면 더 내려갑니다 — 모범답안 참고.


In [ ]:
import pandas as pd, numpy as np
from sklearn.metrics import mean_squared_error
rmse = lambda y, p: mean_squared_error(y, p) ** 0.5

tr = pd.read_csv('data/train_tables.csv')
FEATS = [f'feat_{i}' for i in range(9)] + ['day','hour','minute']
is_val = np.arange(len(tr)) % 5 == 0          # 결정적 held-out (채점과 동일)
a, b = tr[~is_val], tr[is_val]
print('train', len(a), 'val', len(b), '| 결측 비율(평균):', round(tr[FEATS].isna().mean().mean(), 3))


In [ ]:
# BASELINE: 결측을 그대로 둔다 — HistGradientBoosting 은 NaN 을 네이티브로 처리
from sklearn.ensemble import HistGradientBoostingRegressor
m = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.1, random_state=0).fit(a[FEATS], a['target'])
pred = m.predict(b[FEATS])
print('held-out RMSE:', round(rmse(b['target'], pred), 4))

pd.DataFrame({'id': range(len(b)), 'target': pred}).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(b), '(결측 미복원 baseline — 반복 대치로 개선하세요)')


### (선택) 실제 캐글 리더보드 제출용
공식 `test_tables.csv`(컬럼 `id` 포함)에 대한 예측을 `submission_test.csv` 로 저장합니다(캐글 직접 업로드용).
사이트 채점에는 위 `submission.csv`(held-out)만 사용됩니다.

In [ ]:
import os
if os.path.exists('data/test_tables.csv'):
    te = pd.read_csv('data/test_tables.csv')
    pd.DataFrame({'id': te['id'], 'target': m.predict(te[FEATS])}).to_csv('submission_test.csv', index=False)
    print('wrote submission_test.csv', len(te))
else:
    print('data/test_tables.csv 없음 — 캐글 제출용 생략')


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)